In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [110]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
import os
from dotenv import load_dotenv
import copy
load_dotenv()

# Load a sample dataset
from datasets import load_dataset

True

In [3]:
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [96]:
# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

In [97]:
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

In [98]:
tokenizer

GPT2TokenizerFast(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=

Generate with the base model
Here we will try out the base model which does not have a chat template.

In [99]:
# Let's test the base model before training
prompt = "Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print(f"Formatted prompt: {formatted_prompt}")

Formatted prompt: <|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>



In [102]:
# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=30)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Before training:
<|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>

The salt and sugar are both salts, but they are not the same. Salt is a mineral, while sugar is a carbohydrate. Salt is a


In [103]:
ds = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations", split="test")
ds

Dataset({
    features: ['full_topic', 'messages'],
    num_rows: 119
})

In [104]:
ds[0]

{'full_topic': 'Travel/Tourist attractions/Local markets',
 'messages': [{'content': 'Hey!', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm planning a trip to Paris. What are some popular tourist attractions?",
   'role': 'user'},
  {'content': 'The Eiffel Tower, the Louvre Museum, and Notre Dame Cathedral are must-visit places in Paris.',
   'role': 'assistant'},
  {'content': 'That sounds great. Are there any local markets I should check out?',
   'role': 'user'},
  {'content': 'Yes, the Champs-Élysées Christmas Market and the Marché aux Puces de Saint-Ouen (flea market) are very popular among tourists and locals alike.',
   'role': 'assistant'},
  {'content': 'Awesome, thank you for the recommendations!', 'role': 'user'},
  {'content': "You're welcome! Have a great time in Paris!",
   'role': 'assistant'}]}

In [105]:


def format_with_chat_template(example):
    # Convert the list of messages to a single string using the tokenizer's chat template
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return example

# Apply the formatting function to the dataset
ds = ds.map(format_with_chat_template, remove_columns=["messages"])
# ds = ds.rename_column("text", "messages")

In [106]:
ds[0]

{'full_topic': 'Travel/Tourist attractions/Local markets',
 'text': "<|im_start|>user\nHey!<|im_end|>\n<|im_start|>assistant\nHello! How can I help you today?<|im_end|>\n<|im_start|>user\nI'm planning a trip to Paris. What are some popular tourist attractions?<|im_end|>\n<|im_start|>assistant\nThe Eiffel Tower, the Louvre Museum, and Notre Dame Cathedral are must-visit places in Paris.<|im_end|>\n<|im_start|>user\nThat sounds great. Are there any local markets I should check out?<|im_end|>\n<|im_start|>assistant\nYes, the Champs-Élysées Christmas Market and the Marché aux Puces de Saint-Ouen (flea market) are very popular among tourists and locals alike.<|im_end|>\n<|im_start|>user\nAwesome, thank you for the recommendations!<|im_end|>\n<|im_start|>assistant\nYou're welcome! Have a great time in Paris!<|im_end|>\n"}

In [107]:
# print a random example
ds[25]["text"]


"<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\nHello! How can I help you today?<|im_end|>\n<|im_start|>user\nI'm trying to create a budget and I'm not sure how much I should save for emergencies.<|im_end|>\n<|im_start|>assistant\nIt's generally recommended to save 3-6 months' worth of living expenses in an emergency fund.<|im_end|>\n<|im_start|>user\nThat sounds like a lot. How can I get started with saving that much?<|im_end|>\n<|im_start|>assistant\nStart by setting a realistic goal, like saving $1,000 or one month's expenses, and then gradually increase it over time. You can also consider setting aside a fixed amount each month.<|im_end|>\n<|im_start|>user\nOkay, that makes sense. What about shopping – how can I avoid overspending?<|im_end|>\n<|im_start|>assistant\nConsider making a shopping list and sticking to it, avoiding impulse buys, and using the 30-day rule: wait 30 days before buying something non-essential to see if you really need it.<|im_end|>\n"

In [111]:
finetune_name = "SmolLM2-FT-MyDataset"

# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=300,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    eval_strategy="steps",  # Evaluate the model at regular intervals
    eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
   
)

# create a copy of the model to avoid modifying the original
copy_model = copy.deepcopy(model)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=copy_model,
    args=sft_config,
    train_dataset=ds,
    eval_dataset=ds,
)


In [112]:
trainer.train()

/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
50,0.972400,0.820944
100,0.536900,0.471346
150,0.334700,0.268776
200,0.190500,0.161931
250,0.117400,0.114957
300,0.098600,0.100837


/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=300, training_loss=0.48870702266693117, metrics={'train_runtime': 168.0101, 'train_samples_per_second': 7.142, 'train_steps_per_second': 1.786, 'total_flos': 177009724929024.0, 'train_loss': 0.48870702266693117})

In [116]:

outputs = model.generate(**inputs, max_new_tokens=30)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

print("====== Training complete, generating new outputs ======")

outputs = trainer.model.generate(**inputs, max_new_tokens=30)
print("After training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Before training:
<|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>

The salt and sugar are both salts, but they are not the same. Salt is a mineral, while sugar is a carbohydrate. Salt is a
====== Training complete, generating new outputs ======
After training:
<|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>
Yes, one of the main differences between sugar and sugar is its taste. Sugar has a sweet, syrupy consistency, while sugar is more�
